# BetterTogether (Apple Silicon / MPS)

Alternate prompt optimization and weight optimization, evaluate intermediate programs, and retain the best strategy stage.

**Use it when:** You have both a useful prompt optimizer and a trainable local model, and want them to improve each other rather than run in isolation.

**What compilation changes:** Prompt demonstrations first, then a Qwen LoRA adapter in the explicit `p -> w` candidate.

Important configuration in this benchmark:

- BootstrapFewShotWithRandomSearch (`p`) plus BootstrapFinetune (`w`)
- DSPy BetterTogether with explicit `p -> w` strategy
- DSPy LocalProvider with MPS-backed Qwen2.5-0.5B-Instruct, 10 weight-training epochs
- preserve original, `p`, and `p -> w` candidates even when validation ties

Every notebook loads the canonical 300-row expanded dataset and frozen,
pair-grouped membership: 160 training, 60 validation, and 80 locked-test rows.
A semantic human/AI pair can never cross partitions. Optimizer choices use
validation only; the locked test is released once after the program is frozen.
These scores teach optimizer tradeoffs, not a general AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "expanded_notebooks" / "comparison.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'better-together'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='better-together'; live=False
train=160 (human=80, AI=80); validation=60 (human=30, AI=30); test=80 (human=40, AI=40)


## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = dspy.BetterTogether(
    metric=exact_match, p=prompt_optimizer, w=weight_optimizer,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=sol_teacher, valset=valset,
    strategy='p -> w', seed=42,
    optimizer_compile_args={'p': {'teacher': sol_teacher}},
)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 80-row locked test split. The baseline has its own notebook; all other
notebooks report validation and locked-test accuracy separately.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: better-together
task model: Qwen/Qwen2.5-0.5B-Instruct
final test accuracy: 65.0% (52/80)
optimized validation accuracy: 60.0%
same-model baseline: 51.2%
uplift: +13.8 points
accepted traces: human=77, AI=55
optimization cost: $0.8445
optimization time: 1740.0s
mean inference latency: 1.550s
p95 inference latency: 1.922s

Published artifacts:
- program snapshot: chapter06/results/expanded_notebooks/better-together/full/optimized_program.json
- prompt snapshot: chapter06/results/expanded_notebooks/better-together/full/learned_prompt.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Read the validation-selected stage alongside the same-model baseline. If candidates tie or regress, retaining an earlier stage is an honest optimizer outcome, not a reason to consult the locked test.

The saved output above uses the checked-in expanded-dataset result, so opening or
rerunning the notebook is free. The paid run first passed a bounded smoke profile,
then froze its full program using training and validation only. Set
`CHAPTER06_RUN_LIVE=1` before launching Jupyter to reproduce that full protocol;
prompt optimizers require an OpenAI key, while weight optimizers also require the
local PyTorch/Transformers stack. The next cell previews the durable program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 2
1. is_ai=False: Requests will also use custom encodings in the event that you need them. If you have created your own encoding and registered it with the codecs module, you can simply use the c…
2. is_ai=False: If you don't want to use Pydantic models, you can also use Body parameters. See the docs for Body - Multiple Parameters: Singular values in body{.internal-link target=_blank}.

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final test accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`; machine-readable
scores, prompts, programs, predictions, timing, cost, versions, hashes, failures,
and retries live under `results/expanded_notebooks/`. Weight-model payloads are
generated locally and Git-ignored; their checked-in manifests retain file hashes,
sizes, configuration, prompts, programs, and scores. Credentials and provider
request bodies are intentionally excluded.